In [1]:
import os, sys
import zipfile
import pandas as pd
import numpy as np
from IPython.display import clear_output

In [2]:
cwd = os.getcwd()
raw_path = os.path.join(os.path.dirname(cwd),'data','raw','FOB')
zip_files = [f for f in os.listdir(raw_path) if os.path.splitext(f)[1] == '.zip']
display(zip_files)

['XPAR_SHARES_F1_20231002.csv.zip']

In [3]:
for f in zip_files:
    if os.path.splitext(f)[0] not in os.listdir(raw_path):
        with zipfile.ZipFile(os.path.join(raw_path, f), 'r') as zip_ref:
            zip_ref.extractall(raw_path)
        
    df = pd.read_csv(os.path.join(raw_path, os.path.splitext(f)[0]), usecols=['isin'])
    
    #display(df.tolist())    

In [4]:
display(df['isin'].unique()[0])
isin = df['isin'].unique()[0]

'FR0000045072'

In [5]:
chunks = []
for chunk in pd.read_csv(os.path.join(raw_path, os.path.splitext(f)[0]), 
                         header=0, 
                         low_memory=False, 
                         chunksize=10000, 
                         usecols=['isin',
                                   'event_date',
                                   'event_time_cet',
                                   'order_id',
                                   'order_event_type',
                                   'order_side',
                                   'order_price',
                                   'order_size',
                                   'order_type',
                                   'time_in_force', 
                                  'trade_size', 
                                  'trade_price']):
    chunk = chunk[chunk['isin'] == isin]

    chunk['event_time_cet'] = pd.to_datetime(chunk['event_date'] + ' ' + chunk['event_time_cet'])
    chunk = chunk.drop(columns=['event_date'])
    chunks.append(chunk)
    
DF = pd.concat(chunks)

In [6]:
display(DF.sort_index())

,isin,event_time_cet,order_id,order_event_type,order_side,order_price,order_size,order_type,time_in_force,trade_size,trade_price
0,FR0000045072,2023-10-02 03:01:12.559510032,16862367,Reload,Sell,16.50,500.0,Limit,0,0.0,0.0
1,FR0000045072,2023-10-02 03:01:12.559510032,50416806,Reload,Sell,12.80,40.0,Limit,0,0.0,0.0
2,FR0000045072,2023-10-02 03:01:12.559510032,67193907,Reload,Buy,7.13,100.0,Limit,0,0.0,0.0
3,FR0000045072,2023-10-02 03:01:12.559510032,83971245,Reload,Buy,11.34,300.0,Limit,0,0.0,0.0
4,FR0000045072,2023-10-02 03:01:12.559510032,100748461,Reload,Buy,11.34,500.0,Limit,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
679969,FR0000045072,2023-10-02 17:40:00.020208215,23504964784,Cancel,Buy,8.00,0.0,Limit,0,0.0,0.0
679970,FR0000045072,2023-10-02 17:40:00.020208215,1361286614192,Cancel,Buy,8.00,0.0,Limit,0,0.0,0.0
679971,FR0000045072,2023-10-02 17:40:00.020208215,4575767645360,Cancel,Buy,8.00,0.0,Limit,0,0.0,0.0
679972,FR0000045072,2023-10-02 17:40:00.020208215,5231404469424,Cancel,Buy,8.00,0.0,Limit,0,0.0,0.0


In [7]:
ls_id = DF[(DF['order_event_type'] == 'Cancel') | (DF['order_event_type'] == 'Modify')]['order_id'].unique().tolist()

test_shift = DF.copy()

#test_shift['previous_price'] = np.nan

mask = test_shift['order_id'].isin(ls_id)
test_shift.loc[mask, 'previous_price'] = test_shift.loc[mask].groupby('order_id')['order_price'].shift(1)
test_shift.loc[mask, 'previous_size'] = test_shift.loc[mask].groupby('order_id')['order_size'].shift(1)
    
#display(test_shift[test_shift['order_id'] == 23504964784])
display(test_shift)

,isin,event_time_cet,order_id,order_event_type,order_side,order_price,order_size,order_type,time_in_force,trade_size,trade_price,previous_price,previous_size
0,FR0000045072,2023-10-02 03:01:12.559510032,16862367,Reload,Sell,16.50,500.0,Limit,0,0.0,0.0,NaN,NaN
1,FR0000045072,2023-10-02 03:01:12.559510032,50416806,Reload,Sell,12.80,40.0,Limit,0,0.0,0.0,NaN,NaN
2,FR0000045072,2023-10-02 03:01:12.559510032,67193907,Reload,Buy,7.13,100.0,Limit,0,0.0,0.0,NaN,NaN
3,FR0000045072,2023-10-02 03:01:12.559510032,83971245,Reload,Buy,11.34,300.0,Limit,0,0.0,0.0,NaN,NaN
4,FR0000045072,2023-10-02 03:01:12.559510032,100748461,Reload,Buy,11.34,500.0,Limit,0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
679969,FR0000045072,2023-10-02 17:40:00.020208215,23504964784,Cancel,Buy,8.00,0.0,Limit,0,0.0,0.0,8.00,100.0
679970,FR0000045072,2023-10-02 17:40:00.020208215,1361286614192,Cancel,Buy,8.00,0.0,Limit,0,0.0,0.0,8.00,3644.0
679971,FR0000045072,2023-10-02 17:40:00.020208215,4575767645360,Cancel,Buy,8.00,0.0,Limit,0,0.0,0.0,8.00,1.0
679972,FR0000045072,2023-10-02 17:40:00.020208215,5231404469424,Cancel,Buy,8.00,0.0,Limit,0,0.0,0.0,8.00,856.0


In [8]:
time = test_shift['event_time_cet'].sort_values().unique().tolist()[:7]

ret = test_shift.copy()
data = pd.DataFrame(columns=['price', 'size', 'side'])

i=0
l = len(ret)

if 'test.csv' in os.listdir(cwd):
    data = pd.read_csv('test.csv', header=0, index_col=0)
    data.index = pd.to_datetime(data.index)
    last_t = pd.to_datetime(data[-1:].index)
    time = [x for x in time if x > last_t]
    
    if not time:
        data.to_csv('test_def.csv', index=True)
        sys.exit(0)
        
    data = data.loc[last_t]
    


for t in time:
    for _, row in ret[ret['event_time_cet'] == t].iterrows():

        i += 1

        cond = (data['price'] == row['order_price']) & (data['side'] == row['order_side'])
        prev_cond = (data['price'] == row['previous_price']) & (data['side'] == row['order_side'])

        if (row['order_event_type'] == 'Reload') | (row['order_event_type'] == 'New'):

            if len(data[cond]) == 0:
                new_line = pd.Series([row['order_price'], row['order_size'], row['order_side']], index=data.columns.tolist())
                data = pd.concat((data, new_line.to_frame().T), ignore_index=True)

            else:
                data.loc[data[cond].index, 'size'] += row['order_size']

        elif row['order_event_type'] == 'Fill':

            data.loc[data[cond].index, 'size'] -= row['trade_size']

        elif (row['order_event_type'] == 'Modify') | (row['order_event_type'] == 'Cancel'):

            data.loc[data[prev_cond].index, 'size'] -= row['previous_size']

            if len(data[cond]) == 0:
                new_line = pd.Series([row['order_price'], row['order_size'], row['order_side']], index=data.columns.tolist())
                data = pd.concat((data, new_line.to_frame().T), ignore_index=True)

            else:
                data.loc[data[cond].index, 'size'] += row['order_size']

        if i % 500 == 0:
            clear_output(wait=True)
            display(f'{i} / {l}')

    data = data.drop(data[(data == 0).any(axis=1)].index)    
    data = data.sort_values(by=['side','price'])
    data.index = pd.Index([t] * len(data))
    
    if 'test.csv' not in os.listdir(cwd):
        data.to_csv('test.csv', index=True)
    else:
        data.to_csv('test.csv', mode='a', header=False, index=True)
        
    #display(data)
    


SystemExit: 0

C:\Users\Utilisateur\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3351: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
cond = (ret['order_side'] == 'Sell') | (ret['order_side'] == 'Buy')
display(ret[cond])

#test[:1].to_csv('test.csv', index=True)
#test[-1:].to_csv('test.csv', mode='a', header=False, index=True)

In [59]:
display(os.listdir(cwd))

['.ipynb_checkpoints', 'data_FOB_prepro.ipynb', 'ReadMe']

In [80]:
last_t = pd.to_datetime(pd.read_csv('test.csv', header=0, index_col=0)[-1:].index)
display(last_t)

ttt = [x for x in time[:5] if x > last_t]
display(ttt)

for t in time[:5]:
    if t <= last_t:
        display('inf', t)
    else: display('sup', t, last_t)

DatetimeIndex(['2023-10-02 07:30:02.047138394'], dtype='datetime64[ns]', freq=None)

[Timestamp('2023-10-02 07:30:02.047195619')]

'inf'

Timestamp('2023-10-02 03:01:12.559510032')

'inf'

...

'inf'

Timestamp('2023-10-02 07:30:02.047138394')

'sup'

Timestamp('2023-10-02 07:30:02.047195619')

DatetimeIndex(['2023-10-02 07:30:02.047138394'], dtype='datetime64[ns]', freq=None)

In [9]:
ls = [2]

if not ls: display('vide')
else: display('pas vide')

'pas vide'